# 15 · Perfil do eleitorado × voto (correlação ecológica, WLS)

**Desfechos**: % de válidos de PT (nº 13) e PL (nº 22) no 1º turno 2026, sobre válidos do município.

**Preditores** (% do eleitorado do município, grupos disjuntos):
- Idade: `16–17` (derivada), `60–69` (soma nativa), `70+` (derivada). Referência: 18–59. As derivadas **não** são somadas às faixas nativas (sem dupla contagem).
- Sexo: % feminino (referência: masculino).
- Escolaridade: % superior completo e % analfabeto.
- Renda: log10 do **PIB per capita de 2023** (IBGE, mais recente disponível; defasado em relação ao eleitorado de 2026).
- Porte: log10 da **população do Censo 2022** (defasada em relação ao eleitorado de 2026).

**Estimação**: WLS com peso = eleitorado de 2026 do município. IC 95%.

**Ressalvas obrigatórias**: (1) **falácia ecológica**: os coeficientes descrevem médias municipais, não eleitores; (2) **colinearidade**: escolaridade e renda são correlacionadas (seção 3); (3) PIB 2023 e população 2022 não são do mesmo ano do eleitorado de 2026.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path("..") / "src"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
from eleicao.analise import composicao as cp, base, carga, graficos as g
from eleicao.analise import (
    participacao as pa,
    concentracao as cc,
    swing as sw,
    perfil,
    espacial as esp,
    bolsoes as bo,
    mapa_mente as mm,
)

In [ ]:
b = cp.base_2026()
m = b[(b.uf != "zz")].dropna(subset=["pib_per_capita_reais"]).copy()
c26 = cp.candidatos_2026_br()
for nome, nr in (("pt", 13), ("pl", 22)):
    v = c26[c26.nr_candidato == nr][["uf", "cd_mun_tse", "votos"]]
    m = m.merge(v, on=["uf", "cd_mun_tse"], validate="1:1").rename(
        columns={"votos": f"votos_{nome}"}
    )
    m[f"y_{nome}"] = 100 * m[f"votos_{nome}"] / m["validos"]
X = perfil.preparar_variaveis(m)
dados = pd.concat([m[["uf", "cd_mun_tse", "eleitorado", "y_pt", "y_pl"]], X], axis=1)
print("municípios no modelo:", len(dados))

## Coeficientes WLS (modelo básico e completo; o completo acrescenta renda e porte)

In [ ]:
resultados = {}
for y in ("y_pt", "y_pl"):
    for nome, xs in (
        ("básico", perfil.VARIAVEIS_MODELO_BASICO),
        ("completo", perfil.VARIAVEIS_MODELO_COMPLETO),
    ):
        tab, r2, n = perfil.ajustar_wls(dados, y, xs, "eleitorado")
        resultados[(y, nome)] = (tab, r2, n)
        print(f"{y} | modelo {nome} | R² = {r2:.3f} | n = {n}")
        print(tab.round(3).to_string(index=False))
        print()

## VIF (colinearidade entre preditores, modelo completo)

In [ ]:
print(perfil.vif(dados, perfil.VARIAVEIS_MODELO_COMPLETO).round(2).to_string(index=False))

## Colinearidade explícita: escolaridade superior × renda

In [ ]:
mm_ = dados.assign(log_pib=X["log_pib_pc_2023"])
print(pa.correlacao_ecologica(mm_, "pct_grau_superior_completo", "log_pib", "spearman"))

## Gráfico dos coeficientes (modelo completo)

In [ ]:
fig = g.forest_coeficientes(
    {
        "PT (13), completo": resultados[("y_pt", "completo")][0],
        "PL (22), completo": resultados[("y_pl", "completo")][0],
    },
    "WLS ecológico: coeficiente (p.p. de válidos por unidade) e IC 95%",
)
g.salvar(fig, "perfil_coeficientes_pt_pl")
plt.show()